In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
from start_line.plotting import *
from concept_abstraction.training import train_model
from concept_abstraction.selection import greedy_selection, random_selection, human_centered_selection
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ObservationSubsetWrapper
import sys 
import argparse
import secrets
import numpy as np 
import random 
import gymnasium as gym


In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [4]:
if is_jupyter: 
    seed        = 43
    environment_string = "cycle"
    out_folder = "exploration"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string}

In [6]:
np.random.seed(seed)
random.seed(seed)

In [103]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO


# Example usage:

# Pick any subset of indices, e.g. [0, 2] or [1,3] or [0,1,2]
subset_indices = [1,3]  # <- change this as you want!

# Create env and wrap observations
train_env = gym.make("CartPole-v1")
train_env = ObservationSubsetWrapper(train_env, subset_indices)

# Train PPO with default MlpPolicy (input shape auto-detected)
model = PPO("MlpPolicy", train_env, verbose=1)
model.learn(total_timesteps=10_000)

# Evaluation env with rendering and recording
eval_env = gym.make("CartPole-v1", render_mode="rgb_array")
eval_env = ObservationSubsetWrapper(eval_env, subset_indices)
eval_env = gym.wrappers.RecordVideo(eval_env, video_folder="./cartpole-videos", episode_trigger=lambda e: True)

obs, info = eval_env.reset()
total_reward = 0
try:
    for _ in range(500):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward 
        if terminated or truncated:
            break
finally:
    eval_env.close()
    print(total_reward)


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20.9     |
|    ep_rew_mean     | 20.9     |
| time/              |          |
|    fps             | 1102     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 27          |
|    ep_rew_mean          | 27          |
| time/                   |             |
|    fps                  | 866         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009560356 |
|    clip_fraction        | 0.115       |
|    clip_range           | 0.2         |
|    entropy_loss  

Moviepy - Done !
Moviepy - video ready /usr0/home/naveenr/projects/concept_decisions/scripts/notebooks/cartpole-videos/rl-video-episode-0.mp4
444.0


In [27]:
import gymnasium as gym

env = gym.make("CartPole-v1")
states = []
transitions = []  # store (state, action, next_state)

obs, info = env.reset()
for _ in range(1000):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    states.append(obs)
    transitions.append((action, next_obs))
    obs = next_obs
    if terminated or truncated:
        obs, info = env.reset()
    if len(states) >= 1000:
        break



In [28]:
transitions[0]

(1, array([ 0.03396114,  0.15584312, -0.03178483, -0.29326633], dtype=float32))

In [54]:
transition_0 = np.array([i[1] for i in transitions if i[0] == 0])
transition_1 = np.array([i[1] for i in transitions if i[0] == 1])

states_0 = np.array([states[idx] for idx,i in enumerate(transitions) if i[0] == 0])
states_1 = np.array([states[idx] for idx,i in enumerate(transitions) if i[0] == 1])

In [33]:
states = np.array(states)

In [97]:
def get_value(indices):
    all_slopes = []
    for _ in range(100):
        idx_0 = np.random.randint(0,len(states_0))
        idx_1 = np.random.randint(0,len(states_0)) 
        run = np.max(np.abs((states_0[idx_0]-states_0[idx_1])[indices]))
        rise = np.max(np.abs(transition_0[idx_1][1]-transition_0[idx_0][1]))
        
        if run > 0:
            all_slopes.append(rise/run)

    for _ in range(100):
        idx_0 = np.random.randint(0,len(states_1))
        idx_1 = np.random.randint(0,len(states_1)) 
        run = np.max(np.abs((states_1[idx_0]-states_1[idx_1])[indices]))
        rise = np.max(np.abs(transition_1[idx_1][1]-transition_1[idx_0][1]))
        
        if run > 0:
            all_slopes.append(rise/run)
    return np.mean(all_slopes)

In [100]:
total_indices = []

for k in range(4):
    temp_vals = {}
    for val in range(4):
        if val not in total_indices:
            temp_vals[val] = get_value(total_indices+[val])
    min_val = min(temp_vals, key=temp_vals.get)
    total_indices.append(min_val)
    total_indices = sorted(total_indices)
    print(k+1,total_indices,temp_vals[min_val])


1 [1] 1.002248
2 [1, 3] 0.61714345
3 [1, 2, 3] 0.6195039
4 [0, 1, 2, 3] 0.62331015


In [34]:
states

array([[ 0.03475559, -0.03972241, -0.03197143,  0.00933005],
       [ 0.03396114,  0.15584312, -0.03178483, -0.29326633],
       [ 0.03707801,  0.35140347, -0.03765015, -0.5958019 ],
       ...,
       [-0.12154786, -0.22894567,  0.17476147,  0.589907  ],
       [-0.12612678, -0.0366457 ,  0.18655962,  0.3569694 ],
       [-0.1268597 , -0.23386203,  0.193699  ,  0.7021917 ]],
      dtype=float32)

In [ ]:
for _ in range(500):
    action, _ = model.predict(observation, deterministic=True)
    observation, reward, terminated, truncated, info = eval_env.step(action)


## Concept Baseline

In [18]:
values_by_concept = []
baseline_concepts = get_baseline_concept_sets(environment_string)
for concept_list in baseline_concepts:
    print(concept_list)
    env = create_environment_from_string(environment_string,concept_list,0)
    q_net = train_model(env)
    values_by_concept.append(get_values(env,q_net))
results['baseline'] = {
    'concepts': baseline_concepts,
    'values': values_by_concept
}

[0, 1, 2]
[0]
[1]
[2]


## Concept Selection

In [20]:
selected_concepts = []
values_by_random_concept = []
for k in range(1,round(len(env.concepts)**0.5)+1):
    random_concepts = random_selection(env,k)
    env = create_environment_from_string(environment_string,random_concepts,0)
    q_net = train_model(env)
    selected_concepts.append(random_concepts)
    values_by_random_concept.append(get_values(env,q_net))
results['random_selection'] = {
    'concepts': [i.tolist() for i in selected_concepts], 
    'values': values_by_random_concept
}

In [26]:
selected_concepts = []
values_by_greedy_concept = []
for k in range(1,round(len(env.concepts)**0.5)+1):
    greedy_concepts = greedy_selection(env,k)
    env = create_environment_from_string(environment_string,greedy_concepts,0)
    q_net = train_model(env)
    selected_concepts.append(greedy_concepts)
    values_by_greedy_concept.append(get_values(env,q_net))
results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': values_by_greedy_concept
}

In [22]:
concept_list = list(range(env.concepts.shape[0]))
accuracy_by_concept = np.random.random(len(concept_list))
target_abstraction = np.random.random()*0.25

selected_concepts = human_centered_selection(env,accuracy_by_concept,target_abstraction)
selected_concepts = [concept_list[idx] for idx,i in enumerate(selected_concepts) if i>=0.5]
selected_accuracies = [accuracy_by_concept[idx] for idx,i in enumerate(selected_concepts) if i>=0.5]

env = create_environment_from_string(environment_string,selected_concepts,1-np.mean(selected_accuracies))
q_net = train_model(env)
human_perf = get_values(env,q_net)
results['human_selection'] = {
    'accuracies': accuracy_by_concept.tolist(),
    'target': target_abstraction,
    'concepts': selected_concepts,
    'values': values_by_greedy_concept,
}

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-7700K CPU @ 4.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 8 rows, 4 columns and 21 nonzeros
Model fingerprint: 0xb11f5609
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 7e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 8 rows and 4 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.2706760e-01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.270676038e-01


## Performance under Uncertainty

In [23]:
epsilons = [0,0.01,0.1,0.25,0.5]
values_error = [[[] for _ in baseline_concepts[1:]] for _ in epsilons]

for idx,e in enumerate(epsilons):
    for jdx,concept_list in enumerate(baseline_concepts[1:]):
        env = create_environment_from_string(environment_string,concept_list,e)
        q_net = train_model(env)
        values_error[idx][jdx] = get_values(env,q_net)
results['uncertainty'] = {
    'epsilons': epsilons, 
    'concepts': baseline_concepts[1:],
    'values': values_error,
}

KeyboardInterrupt: 

## Save Data

In [ ]:
save_path = get_save_path(out_folder,save_name)

In [ ]:
delete_duplicate_results(out_folder,"",results)

In [ ]:
json.dump(results,open('../../results/'+save_path,'w'))